In [ ]:
## Phase 3 — Data Preprocessing

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

In [2]:
df = pd.read_csv('data/dataset.csv')


In [3]:
df.columns = df.columns.str.strip()
symptom_cols = [col for col in df.columns if col.startswith('Symptom')]

all_symptoms = []
for col in symptom_cols:
    all_symptoms.extend(df[col].dropna().astype(str).tolist())
all_symptoms = sorted(set([s.strip().lower() for s in all_symptoms]))

severity_df = pd.read_csv('data/Symptom-severity.csv')
severity_map = dict(zip(
    severity_df['Symptom'].astype(str).str.strip().str.lower(),
    severity_df['weight']
))

binary_df = pd.DataFrame(0, index=df.index, columns=all_symptoms)
for col in symptom_cols:
    for idx, val in df[col].dropna().items():
        symptom = str(val).strip().lower()
        binary_df.at[idx, symptom] = severity_map.get(symptom, 1)

binary_df['disease'] = df['Disease'].str.strip()

In [4]:
le = LabelEncoder()
binary_df['disease_encoded'] = le.fit_transform(binary_df['disease'])


In [5]:
X = binary_df[all_symptoms]
y = binary_df['disease_encoded']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)


## Phase 4 — Model Training

In [6]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import GridSearchCV

dt = DecisionTreeClassifier(random_state=42)
rf = RandomForestClassifier(random_state=42)
nb = GaussianNB()

dt.fit(X_train, y_train)
rf.fit(X_train, y_train)
nb.fit(X_train, y_train)

params = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5]
}

grid = GridSearchCV(RandomForestClassifier(random_state=42), params, cv=5, scoring='accuracy')
grid.fit(X_train, y_train)
rf_best = grid.best_estimator_

print('Best Parameters:', grid.best_params_)
print('Best CV Accuracy:', grid.best_score_)

voting_clf = VotingClassifier(
    estimators=[('dt', dt), ('rf', rf_best), ('nb', nb)],
    voting='soft'
 )
voting_clf.fit(X_train, y_train)

Best Parameters: {'max_depth': None, 'min_samples_split': 2, 'n_estimators': 100}
Best CV Accuracy: 1.0


,"estimators estimators: list of (str, estimator) tuplesInvoking the ``fit`` method on the ``VotingClassifier`` will fit clonesof those original estimators that will be stored in the class attribute``self.estimators_``. An estimator can be set to ``'drop'`` using:meth:`set_params`... versionchanged:: 0.21 ``'drop'`` is accepted. Using None was deprecated in 0.22 and support was removed in 0.24.","[('dt', ...), ('rf', ...), ...]"
,"voting voting: {'hard', 'soft'}, default='hard'If 'hard', uses predicted class labels for majority rule voting.Else if 'soft', predicts the class label based on the argmax ofthe sums of the predicted probabilities, which is recommended foran ensemble of well-calibrated classifiers.",'soft'
,"weights weights: array-like of shape (n_classifiers,), default=NoneSequence of weights (`float` or `int`) to weight the occurrences ofpredicted class labels (`hard` voting) or class probabilitiesbefore averaging (`soft` voting). Uses uniform weights if `None`.",None
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel for ``fit``.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionadded:: 0.18",None
,"flatten_transform flatten_transform: bool, default=TrueAffects shape of transform output only when voting='soft'If voting='soft' and flatten_transform=True, transform method returnsmatrix with shape (n_samples, n_classifiers * n_classes). Ifflatten_transform=False, it returns(n_classifiers, n_samples, n_classes).",True
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting will be printed as itis completed... versionadded:: 0.23",False
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.",'gini'
,"splitter splitter: {""best"", ""random""}, default=""best""The strategy used to choose the split at each node. Supportedstrategies are ""best"" to choose the best split and ""random"" to choosethe best random split.",'best'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1


## Phase 5 — Model Evaluation

In [7]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns


In [8]:
from sklearn.metrics import accuracy_score

models = {
    'Decision Tree': dt,
    'Random Forest': rf_best,
    'Naive Bayes': nb,
    'Voting Classifier': voting_clf,
}

for name, model in models.items():
    preds = model.predict(X_test)
    print(f"{name} Accuracy: {accuracy_score(y_test, preds):.4f}")

Decision Tree Accuracy: 1.0000
Random Forest Accuracy: 1.0000
Naive Bayes Accuracy: 1.0000
Voting Classifier Accuracy: 1.0000


In [ ]:
for name, model in models.items():
    preds = model.predict(X_test)
    cm = confusion_matrix(y_test, preds)
    plt.figure(figsize=(14, 10))
    sns.heatmap(cm, annot=True, fmt='d', xticklabels=le.classes_, yticklabels=le.classes_)
    plt.title(f'Confusion Matrix - {name}')
    plt.tight_layout()
    plt.show()


In [ ]:
importances = rf_best.feature_importances_
top_idx = np.argsort(importances)[-15:]
plt.figure(figsize=(10, 6))
plt.barh([all_symptoms[i] for i in top_idx], importances[top_idx])
plt.title('Top 15 Most Predictive Symptoms (Tuned RF)')
plt.tight_layout()
plt.show()

In [9]:
results = {}
for name, model in {'Decision Tree': dt, 'Random Forest': rf_best, 'Naive Bayes': nb, 'Voting Classifier': voting_clf}.items():
    preds = model.predict(X_test)
    results[name] = round(accuracy_score(y_test, preds) * 100, 2)

summary = pd.DataFrame(results.items(), columns=['Model', 'Accuracy (%)'])
print(summary.sort_values('Accuracy (%)', ascending=False))

rf_best_acc = accuracy_score(y_test, rf_best.predict(X_test))
voting_acc = accuracy_score(y_test, voting_clf.predict(X_test))
best_model = voting_clf if voting_acc > rf_best_acc else rf_best
best_name = 'Voting Classifier' if voting_acc > rf_best_acc else 'Random Forest (Tuned)'
print('Saved model choice:', best_name)

               Model  Accuracy (%)
0      Decision Tree         100.0
1      Random Forest         100.0
2        Naive Bayes         100.0
3  Voting Classifier         100.0
Saved model choice: Random Forest (Tuned)


In [13]:
# External holdout evaluation using the Kaggle Training.csv / Testing.csv split
external_train = pd.read_csv('../Training.csv')
external_test = pd.read_csv('../Testing.csv')

external_train.columns = external_train.columns.str.strip()
external_test.columns = external_test.columns.str.strip()

def build_external_feature_frame(source_frame, target_features):
    feature_frame = pd.DataFrame(0, index=source_frame.index, columns=target_features)
    for feature in target_features:
        matching_columns = [col for col in source_frame.columns if col == feature or col == f'{feature}.1']
        if not matching_columns:
            continue
        combined = source_frame[matching_columns].astype(int).max(axis=1)
        feature_frame[feature] = combined * severity_map.get(feature, 1)
    return feature_frame

external_test_X = build_external_feature_frame(external_test, all_symptoms)
external_test_y = le.transform(external_test['prognosis'].astype(str).str.strip())

external_models = {
    'Decision Tree': dt,
    'Random Forest': rf_best,
    'Naive Bayes': nb,
    'Voting Classifier': voting_clf,
}

external_results = {}
for name, model in external_models.items():
    preds = model.predict(external_test_X)
    external_results[name] = round(accuracy_score(external_test_y, preds) * 100, 2)

external_summary = pd.DataFrame(external_results.items(), columns=['Model', 'Accuracy (%)'])
print('External holdout accuracy on Testing.csv:')
print(external_summary.sort_values('Accuracy (%)', ascending=False))

External holdout accuracy on Testing.csv:
               Model  Accuracy (%)
0      Decision Tree         100.0
1      Random Forest         100.0
2        Naive Bayes         100.0
3  Voting Classifier         100.0


## Phase 6 — Save Model and Assets

In [10]:
import pickle

with open('model.pkl', 'wb') as f:
    pickle.dump(best_model, f)

with open('columns.pkl', 'wb') as f:
    pickle.dump(all_symptoms, f)

with open('label_encoder.pkl', 'wb') as f:
    pickle.dump(le, f)

## Phase 7 — Prediction Pipeline

In [11]:
def predict_disease(selected_symptoms):
    selected_symptoms = [s.strip().lower() for s in selected_symptoms]
    if not selected_symptoms or len(selected_symptoms) < 3:
        return 'Please select at least 3 symptoms for a meaningful prediction.'

    input_vector = [severity_map.get(s, 1) if s in selected_symptoms else 0 for s in all_symptoms]
    input_df = pd.DataFrame([input_vector], columns=all_symptoms)

    proba = best_model.predict_proba(input_df)[0]
    top3_idx = np.argsort(proba)[-3:][::-1]

    results = []
    for idx in top3_idx:
        disease = le.classes_[idx]
        confidence = round(proba[idx] * 100, 2)
        results.append((disease, confidence))

    return results